# 面试问题：Activation Checkpointing 为什么省显存？重计算、随机性、状态副作用和分段怎样实现？

**一句话回答**：普通反向传播保存每层激活；checkpointing 只保存少量边界，backward 到某段时重跑 forward 恢复中间激活，以额外计算换峰值显存。它不等于训练 checkpoint；正确实现还要复现 dropout RNG、避免 BatchNorm/计数器副作用重复更新，并按层成本和显存预算选择分段。

本 Notebook 用 NumPy 手写一条 tanh 网络的完整缓存与分段重计算反向，比较参数梯度，并实现内存模型、确定性随机 mask、副作用门禁和计划搜索。每个代码单元都有中文关键注释。


In [ ]:
import hashlib,math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 使用 float64 建立高精度梯度 oracle。
SEED141=14101; rng141=np.random.default_rng(SEED141)  # 计算并保存当前步骤的中间状态。
assert SEED141==14101  # 用受控断言验证关键不变量。
assert np.tanh(0)==0  # 用受控断言验证关键不变量。
assert np.isfinite(rng141.normal())  # 用受控断言验证关键不变量。


## 1. 区分参数状态、激活和临时 workspace

activation 随 micro-batch、sequence、hidden、层数增长，训练反向需要保存输入/输出等张量；参数、梯度和 Adam 状态则由参数量决定。重计算主要削减 saved activations，不会自动减少 optimizer/KV/cache，也可能增加临时峰值。


In [ ]:
def activation_bytes141(batch,seq,hidden,layers,bytes_=2,tensors_per_layer=8):  # 定义本节可复用的核心函数。
    # 这是容量下界模型，不包含 allocator 碎片与 kernel workspace。
    return batch*seq*hidden*layers*bytes_*tensors_per_layer  # 返回当前分支计算出的结果。
base_mem141=activation_bytes141(2,2048,4096,32)  # 计算并保存当前步骤的中间状态。
assert base_mem141>1e9  # 用受控断言验证关键不变量。
assert activation_bytes141(4,2048,4096,32)==2*base_mem141  # 用受控断言验证关键不变量。
assert activation_bytes141(2,4096,4096,32)==2*base_mem141  # 用受控断言验证关键不变量。


## 2. 均匀分段只保存边界，段内 backward 前重算

L 层每 k 层一个边界，长期保存约 `ceil(L/k)+1` 个边界，重算时临时保留最多 k 层。简化峰值与 `L/k+k` 同阶，在 `k≈√L` 附近较小；真实 Transformer 应按每层激活字节与计算耗时非均匀切分。


In [ ]:
def segment_bounds141(layers,k):  # 定义本节可复用的核心函数。
    # 末段不足 k 层也必须把 L 作为最终边界。
    out=list(range(0,layers,k))  # 计算并保存当前步骤的中间状态。
    return out+([layers] if not out or out[-1]!=layers else [])  # 返回当前分支计算出的结果。
bounds141=segment_bounds141(10,3)  # 计算并保存当前步骤的中间状态。
assert bounds141==[0,3,6,9,10]  # 用受控断言验证关键不变量。
assert len(segment_bounds141(16,4))==5  # 用受控断言验证关键不变量。
assert segment_bounds141(1,8)==[0,1]  # 用受控断言验证关键不变量。


## 3. 先建立保存全部激活的反向 oracle

链式网络 `a_{i+1}=tanh(w_i a_i+b_i)` 保存所有 `a_i`。反向使用 `1-a²` 求局部导数，得到每层 `dw/db` 和输入梯度。这个实现占用 O(L) 激活，但为重计算版本提供明确数值基线。


In [ ]:
def full_backward141(ws,bs,x):  # 定义本节可复用的核心函数。
    # forward 保存每层输出，backward 直接倒序读取。
    acts=[float(x)]  # 计算并保存当前步骤的中间状态。
    for w,b in zip(ws,bs): acts.append(float(np.tanh(w*acts[-1]+b)))  # 遍历输入元素以累积或检查结果。
    gw=np.zeros_like(ws); gb=np.zeros_like(bs); upstream=1.0  # 计算并保存当前步骤的中间状态。
    for i in reversed(range(len(ws))):  # 遍历输入元素以累积或检查结果。
        dz=upstream*(1-acts[i+1]**2); gw[i]=dz*acts[i]; gb[i]=dz; upstream=dz*ws[i]  # 计算并保存当前步骤的中间状态。
    return acts[-1],gw,gb,upstream,acts  # 返回当前分支计算出的结果。
ws141=rng141.normal(.8,.1,8); bs141=rng141.normal(0,.05,8); full141=full_backward141(ws141,bs141,.7)  # 计算并保存当前步骤的中间状态。
assert len(full141[-1])==9  # 用受控断言验证关键不变量。
assert np.all(np.isfinite(full141[1]))  # 用受控断言验证关键不变量。
assert full141[0]==full141[-1][-1]  # 用受控断言验证关键不变量。


## 4. 分段 backward 从边界重新 forward

首次 forward 只持久保存 segment boundaries。反向逐段倒序：从该段起点重算局部 activations，完成局部梯度后立即释放，再进入上一段。下面验证输出、每层参数梯度和输入梯度都与完整缓存一致。


In [ ]:
def checkpoint_backward141(ws,bs,x,k):  # 定义本节可复用的核心函数。
    bounds=segment_bounds141(len(ws),k); checkpoints={0:float(x)}; a=float(x)  # 计算并保存当前步骤的中间状态。
    # 首次 forward 仅在边界处留下激活。
    for i,(w,b) in enumerate(zip(ws,bs),1):  # 遍历输入元素以累积或检查结果。
        a=float(np.tanh(w*a+b))  # 计算并保存当前步骤的中间状态。
        if i in bounds: checkpoints[i]=a  # 按当前条件选择后续控制路径。
    gw=np.zeros_like(ws); gb=np.zeros_like(bs); upstream=1.0; recomputed=0  # 计算并保存当前步骤的中间状态。
    for start,end in reversed(list(zip(bounds[:-1],bounds[1:]))):  # 遍历输入元素以累积或检查结果。
        local=[checkpoints[start]]  # 计算并保存当前步骤的中间状态。
        for i in range(start,end): local.append(float(np.tanh(ws[i]*local[-1]+bs[i]))); recomputed+=1  # 遍历输入元素以累积或检查结果。
        for i in reversed(range(start,end)):  # 遍历输入元素以累积或检查结果。
            j=i-start; dz=upstream*(1-local[j+1]**2); gw[i]=dz*local[j]; gb[i]=dz; upstream=dz*ws[i]  # 计算并保存当前步骤的中间状态。
    return a,gw,gb,upstream,checkpoints,recomputed  # 返回当前分支计算出的结果。
ck141=checkpoint_backward141(ws141,bs141,.7,3)  # 计算并保存当前步骤的中间状态。
assert np.allclose(ck141[1],full141[1])  # 用受控断言验证关键不变量。
assert np.allclose(ck141[2],full141[2])  # 用受控断言验证关键不变量。
assert math.isclose(ck141[3],full141[3],rel_tol=1e-12)  # 用受控断言验证关键不变量。


## 5. Dropout 等随机算子必须在重算时得到同一 mask

若重算消耗新的全局 RNG，backward 对应的函数已不是原 forward。可保存/恢复 RNG state，或使用由 global step、micro-batch、layer、元素坐标决定的 counter-based RNG。后者不依赖分段顺序，更适合并行重排。


In [ ]:
def keep141(seed,step,micro,layer,index,p=.2):  # 定义本节可复用的核心函数。
    # 哈希坐标模拟 counter-based RNG，重算顺序改变也不变。
    raw=f"{seed}:{step}:{micro}:{layer}:{index}".encode(); u=int.from_bytes(hashlib.sha256(raw).digest()[:8],"big")/2**64  # 计算并保存当前步骤的中间状态。
    return u>=p  # 返回当前分支计算出的结果。
mask_a141=[keep141(9,10,2,3,i) for i in range(32)]; mask_b141=[keep141(9,10,2,3,i) for i in reversed(range(32))][::-1]  # 计算并保存当前步骤的中间状态。
assert mask_a141==mask_b141  # 用受控断言验证关键不变量。
assert any(mask_a141) and not all(mask_a141)  # 用受控断言验证关键不变量。
assert keep141(9,10,2,3,5)==keep141(9,10,2,3,5)  # 用受控断言验证关键不变量。


## 6. 重算区域必须是可重入的纯计算

BatchNorm running stats、随机采样器、日志计数、缓存写入和外部 I/O 若在第二次 forward 再执行，会重复副作用。把状态更新移到 checkpoint 区域外，或给重算上下文显式 `is_recompute` 语义；不能用“结果差不多”掩盖状态漂移。


In [ ]:
class CounterLayer141:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.updates=0  # 定义本节可复用的核心函数。
    def forward(self,x,is_recompute=False):  # 定义本节可复用的核心函数。
        # 只有真实 forward 更新持久状态，重算只恢复数值图。
        if not is_recompute: self.updates+=1  # 按当前条件选择后续控制路径。
        return x*2  # 返回当前分支计算出的结果。
layer141=CounterLayer141(); y141=layer141.forward(3); y_re141=layer141.forward(3,is_recompute=True)  # 计算并保存当前步骤的中间状态。
assert y141==y_re141==6  # 用受控断言验证关键不变量。
assert layer141.updates==1  # 用受控断言验证关键不变量。
assert layer141.forward(1,is_recompute=True)==2 and layer141.updates==1  # 用受控断言验证关键不变量。


## 7. 按显存预算搜索 segment，而不是固定“每 N 层”

简化代价包含持久边界和段内临时激活，计算开销由被重算层 FLOPs 决定。异构层、MoE、长 attention 应用动态规划或 profile 数据切段。预算不可行时还需结合 micro-batch、sequence parallel、offload 或更小模型。


In [ ]:
def plan_cost141(L,k,act_mb,layer_ms):  # 定义本节可复用的核心函数。
    # 峰值近似为边界数加最大段长，额外计算约重跑一次全部层。
    peak=(math.ceil(L/k)+1+k)*act_mb; extra=L*layer_ms  # 计算并保存当前步骤的中间状态。
    return peak,extra  # 返回当前分支计算出的结果。
candidates141=[(k,*plan_cost141(32,k,100,2)) for k in range(1,17)]; feasible141=[x for x in candidates141 if x[1]<=1400]  # 计算并保存当前步骤的中间状态。
best141=min(feasible141,key=lambda x:(x[2],x[1]))  # 计算并保存当前步骤的中间状态。
assert feasible141  # 用受控断言验证关键不变量。
assert best141[1]<=1400  # 用受控断言验证关键不变量。
assert min(x[1] for x in candidates141)<plan_cost141(32,1,100,2)[0]  # 用受控断言验证关键不变量。


## 8. 验收同时看峰值显存、吞吐和梯度等价

在固定 seed 下比较 checkpoint on/off 的 loss、参数梯度与 optimizer step；覆盖 dropout、autocast、非整除段、共享参数、in-place 操作和分布式 collective。记录实际 peak allocated/reserved、step time 与重算比例，理论节省不能代替 profiler。


In [ ]:
report141={"layers":len(ws141),"segment":3,"saved_boundaries":len(ck141[4]),"recomputed_layers":ck141[5],"max_grad_error":float(np.max(np.abs(ck141[1]-full141[1])))}  # 计算并保存当前步骤的中间状态。
# 受控链路要求每层重算一次且梯度误差接近机器精度。
assert report141["recomputed_layers"]==len(ws141)  # 用受控断言验证关键不变量。
assert report141["saved_boundaries"]<len(full141[-1])  # 用受控断言验证关键不变量。
assert report141["max_grad_error"]<1e-12  # 用受控断言验证关键不变量。


## 面试总结

完整回答是：**拆分模型状态与 activation → 选择边界而非保存每层 → backward 分段重跑 forward → 与完整缓存做输出/梯度 oracle → RNG 按坐标复现 → 状态副作用只执行一次 → 按真实层显存/FLOPs 搜段 → 结合 micro-batch/并行 → 用 profiler 验证峰值和吞吐**。Activation checkpointing 是计算换显存，不是保存训练断点。

延伸阅读：[Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174)、[GPipe](https://arxiv.org/abs/1811.06965)、[PyTorch Checkpoint 文档](https://pytorch.org/docs/stable/checkpoint.html)。
